# 200M FineWeb-Edu: mLSTM + PPLSTM vs parameter-matched Transformer

**Two models, trained from scratch on the same FineWeb-Edu (`cartman`) token batches:**

- **Hybrid:** 4 official xLSTM residual blocks in the order **mLSTM → PPLSTM → mLSTM → PPLSTM**, retaining the original published PPLSTM cell and the working CUDA Graph forward+backward approach.
- **Transformer:** 4 pre-LayerNorm causal attention blocks, embedding/output weights tied; feed-forward hidden size chosen automatically to **match total trainable parameters** as closely as possible. PyTorch **scaled-dot-product attention (SDPA)** uses fused attention kernels when supported (and automatically falls back with an explicit note if unavailable). We use GPU **FP16 attention calculations only** (Q/K/V attention computation) while keeping the rest of both models FP32. This is documented as an attention-precision difference.

**Budget:** up to **200 million scored training-token presentations per model** (400M total), **not** 200M unique tokens. The sampler reuses the attached corpus. Run has a time guard (default 5.7 hours), alternating training chunks and saving resumable checkpoints. Same optimizer, learning-rate schedule, tokens and validation set. Neither model uses an sLSTM block.

**Optional post-training tests:** separately trained models on *parity*, *four-state tracking*, *delayed recall*, *key-value recall*, *key-value updates*, *two-hop* and *three-hop* associative chaining. These are supervised synthetic-task probes and do **not** establish zero-shot reasoning ability of the language models. They run only when time remains.

**Important:** PyTorch CUDA Graphs are *not* custom fused kernels. The PPLSTM implementation is unchanged. SDPA may choose Flash, memory-efficient, or math backend depending on the T4, PyTorch and dtype; the fused availability check and backend choice are logged. If Kaggle offers a GPU other than T4, results describe the actual device. This notebook is a fresh experiment; no older scores are loaded. The current Kaggle package installation requires Internet enabled.

## 1. Settings, hardware, dependencies and output

In [ ]:
# EDIT THESE FIRST.
TARGET_TOKENS_PER_MODEL = 200_000_000
WIDTH, DEPTH, HEADS = 72, 4, 4
CONTEXT, BATCH = 128, 8
SEED = 20261116
CHUNK_TOKENS = 5_000_000            # alternate the models at least every 5M presentations
VALIDATE_EVERY = 10_000_000         # score held-out tokens every 10M
SAVE_EVERY = 10_000_000             # two model + optimizer checkpoints
MAX_RUN_HOURS = 5.7                 # 6.3 h reported remaining: leave >30 min buffer
SMALL_VALIDATION_TOKENS = 8_192
FINAL_EVAL_TOKENS = 65_536
MAX_PREP_TRAIN_TOKENS = 64_000_000  # distinct prepared BPE tokens, NOT the 200M training presentations
TRAIN_PARQUET_GLOB = '/kaggle/input/**/000_00000.parquet'  # attach 'cartman'
PREPARED_DIR = ''                     # optional directory with train.bin,valid.bin,test.bin,spm.model
RESUME_INPUT_DIR = ''                 # optional previous-output folder with both checkpoints
# The official xLSTM dependency is required for the mLSTM layers.
PPLSTM_CUDA_GRAPH_REQUIRED = True
TRANSFORMER_CUDA_GRAPH_REQUIRED = False
USE_FP16_SDPA_ATTENTION = True       # fast attention only; attention tensor math half, rest FP32
FUSED_SDPA_ONLY = False             # if True, fail rather than allow a math-kernel fallback
RUN_PROBES = True
PROBE_STEPS = 1_500
PROBE_BATCH = 16
PROBE_TRAIN_CONTEXT = 128
PROBE_EVAL_CONTEXTS = (128,512)
PROBE_EVAL_EXAMPLES = 512
PROBE_REPORT_EVERY = 500
STARTUP_ONLY = False                # True: just checks, model matching and a 50-step speed trial

from pathlib import Path
import os,sys,subprocess,importlib.util,time,random,math,statistics,copy,glob,json,hashlib,zipfile,traceback
OUT=Path('/kaggle/working/pplstm_vs_transformer_200m' if Path('/kaggle/working').exists() else '/mnt/data/pplstm_vs_transformer_200m')
OUT.mkdir(parents=True,exist_ok=True)
need=[]
if importlib.util.find_spec('xlstm') is None: need.append('xlstm==2.0.6')
if importlib.util.find_spec('sentencepiece') is None: need.append('sentencepiece')
if importlib.util.find_spec('pyarrow') is None: need.append('pyarrow')
if need:
    print('Installing required libraries:',need)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
assert torch.cuda.is_available(), 'Kaggle Settings → Accelerator → GPU required; CPU would be much too slow.'
DEVICE=torch.device('cuda:0')
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
print('CUDA:',torch.cuda.get_device_name(0),'compute capability:',torch.cuda.get_device_capability(0),'PyTorch:',torch.__version__)
RUN_START=time.monotonic();DEADLINE=RUN_START+MAX_RUN_HOURS*3600

def write_json(path,data):
    path=Path(path);temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(data,indent=2,default=str));os.replace(temp,path)
def save_table(name,rows):
    if rows:pd.DataFrame(rows).to_csv(OUT/name,index=False)
def atomic_save(obj,path):
    path=Path(path);tmp=path.with_suffix('.pt.tmp');torch.save(obj,tmp);os.replace(tmp,path)
def bundle():
    name=OUT/'PPLSTM_vs_Transformer_FineWeb_200M_Results.zip';temp=OUT/'results.partial.zip'
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED) as z:
        for item in [*OUT.glob('*.json'),*OUT.glob('*.csv')]:z.write(item,item.name)
    os.replace(temp,name)
    return name
def nparams(model):return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 2. Original PPLSTM cell — equations and initialization retained exactly

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Adapted from the original PPLSTM project by Rickey (https://github.com/rickey1990/PPLSTM/blob/main/pplstm.py)
# The transition equations and scan implementation are kept as published.
"""PPLSTM: associative 3-channel permutation-mixed recurrent cell."""
import itertools
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F


def shift_right(x: Tensor) -> Tensor:
    return torch.cat((torch.zeros_like(x[:, :1]), x[:, :-1]), dim=1)


def scalar_affine_scan(a: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    A, B = a, b
    step = 1
    while step < a.shape[1]:
        A, B = (
            torch.cat((A[:, :step], A[:, step:] * A[:, :-step]), dim=1),
            torch.cat((B[:, :step], B[:, step:] + A[:, step:] * B[:, :-step]), dim=1),
        )
        step *= 2
    return A, B


def matrix_affine_scan(A: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    step = 1
    while step < A.shape[1]:
        next_A = torch.cat((A[:, :step], A[:, step:] @ A[:, :-step]), dim=1)
        next_b = torch.cat((
            b[:, :step],
            b[:, step:] + (A[:, step:] @ b[:, :-step].unsqueeze(-1)).squeeze(-1),
        ), dim=1)
        A, b = next_A, next_b
        step *= 2
    return A, b


class PPLSTM(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        if width < 3 or width % 3 != 0:
            raise ValueError('width must be a positive multiple of 3')
        self.width = width
        self.groups = width // 3
        self.xproj = nn.Linear(width, 4 * width)
        self.mix_logit = nn.Parameter(torch.full((4, width), math.log(3.0)))
        self.sign_proj = nn.Linear(width, width)
        self.initial_cell = nn.Parameter(torch.linspace(-1.5, 1.5, 3).repeat(self.groups))
        self.input_bias = nn.Parameter(torch.full((width,), -4.0))
        self.forget_bias = nn.Parameter(torch.full((width,), 4.0))
        nn.init.zeros_(self.sign_proj.weight)
        nn.init.constant_(self.sign_proj.bias, 2.8)

        bank = []
        for order in itertools.permutations(range(3)):
            matrix = torch.zeros(3, 3)
            for row, column in enumerate(order):
                matrix[row, column] = 1.0
            bank.append(matrix)
        self.register_buffer('permutation_bank', torch.stack(bank))

        self.selector = nn.Linear(width, self.groups * 6)
        nn.init.zeros_(self.selector.weight)
        nn.init.constant_(self.selector.bias, -1.0)
        with torch.no_grad():
            for group in range(self.groups):
                self.selector.bias[group * 6] = 3.0

    def _token_maps(self, x: Tensor):
        if x.ndim != 3 or x.shape[-1] != self.width:
            raise ValueError('Expected x with shape [batch, time, width]')
        batch, length, width = x.shape
        group_count = self.groups
        previous = shift_right(x)
        coeff = self.mix_logit.sigmoid().view(1, 1, 4, width)
        gates = (self.xproj(x).reshape(batch, length, 4, width) * coeff
                 + self.xproj(previous).reshape(batch, length, 4, width) * (1 - coeff))
        input_logits, forget_logits, candidate_logits, output_logits = gates.unbind(2)
        mixed = x * self.mix_logit[0].sigmoid() + previous * (1 - self.mix_logit[0].sigmoid())
        signs = self.sign_proj(mixed).tanh().reshape(batch, length, group_count, 3)
        z = candidate_logits.tanh().reshape(batch, length, group_count, 3)
        log_i = (input_logits + self.input_bias).reshape(batch, length, group_count, 3).mean(-1)
        log_f = F.logsigmoid((forget_logits + self.forget_bias)
                           .reshape(batch, length, group_count, 3).mean(-1))
        f_prefix = log_f.cumsum(dim=1)
        m = f_prefix + torch.cummax(torch.cat((
            torch.zeros_like(f_prefix[:, :1]), log_i - f_prefix,
        ), dim=1), dim=1).values[:, 1:]
        f = (log_f + shift_right(m) - m).exp()
        i = (log_i - m).exp()
        weights = self.selector(mixed).reshape(batch, length, group_count, 6).softmax(-1)
        P = torch.einsum('btgp,pij->btgij', weights, self.permutation_bank)
        A = f[..., None, None] * P * signs[..., None, :]
        b = i[..., None] * z
        return A, b, f, i, output_logits

    def forward(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        prefix_A, prefix_b = matrix_affine_scan(A, b)
        c0 = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        c = (prefix_A @ c0[:, None, :, :, None]).squeeze(-1) + prefix_b
        norm_A, norm_b = scalar_affine_scan(f, i)
        n = norm_A + norm_b
        ratio = (c / n[..., None].clamp_min(1e-12)).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

    def serial_reference(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        c = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        n = x.new_ones(batch, self.groups)
        states = []
        for t in range(length):
            c = (A[:, t] @ c.unsqueeze(-1)).squeeze(-1) + b[:, t]
            n = f[:, t] * n + i[:, t]
            states.append(c / n[..., None].clamp_min(1e-12))
        ratio = torch.stack(states, dim=1).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

## 3. Four-block hybrid and fast-attention Transformer

In [ ]:
from xlstm import (xLSTMBlockStack, xLSTMBlockStackConfig, mLSTMBlockConfig,
                   mLSTMLayerConfig, sLSTMBlockConfig, sLSTMLayerConfig, FeedForwardConfig)

class PPLSTMReplacementLayer(nn.Module):
    """Adapt original sLSTM layer slot; retain convolution, group norm and dropout."""
    def __init__(self, original_layer):
        super().__init__()
        self.config = original_layer.config
        self.conv1d = original_layer.conv1d if self.config.conv1d_kernel_size > 0 else None
        self.conv_act_fn = original_layer.conv_act_fn if self.conv1d is not None else None
        self.group_norm = original_layer.group_norm
        self.dropout = original_layer.dropout
        self.pplstm = PPLSTM(self.config.embedding_dim)

    def forward(self, x, **kwargs):
        B,S,W=x.shape
        conv_x=self.conv_act_fn(self.conv1d(x)) if self.conv1d is not None else x
        y=self.dropout(self.pplstm(conv_x))
        H=self.config.num_heads
        y=y.reshape(B,S,H,W//H).transpose(1,2)
        y=self.group_norm(y).transpose(1,2).reshape(B,S,W)
        return y


def build_backbone(width=WIDTH, context=CONTEXT):
    assert width%3==0 and width%HEADS==0
    cfg=xLSTMBlockStackConfig(
        mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig(
            conv1d_kernel_size=4, qkv_proj_blocksize=4, num_heads=HEADS)),
        slstm_block=sLSTMBlockConfig(slstm=sLSTMLayerConfig(
            backend='vanilla', num_heads=HEADS, conv1d_kernel_size=4,
            bias_init='powerlaw_blockdependent'),
            feedforward=FeedForwardConfig(proj_factor=1.3, act_fn='gelu')),
        context_length=context, num_blocks=DEPTH,
        embedding_dim=width, slstm_at=[1,3])
    stack=xLSTMBlockStack(cfg)
    assert list(stack.config.block_map)==[0,1,0,1]
    for index in (1,3):
        original=stack.blocks[index].xlstm
        if not hasattr(original,'slstm_cell'):
            raise RuntimeError('xLSTM API changed: expected slstm_cell in layer '+str(index))
        stack.blocks[index].xlstm=PPLSTMReplacementLayer(original)
    return stack

class PPLSTMLanguageModel(nn.Module):
    def __init__(self, vocab, width=WIDTH, context=CONTEXT):
        super().__init__()
        self.vocab=vocab
        self.emb=nn.Embedding(vocab,width)
        nn.init.normal_(self.emb.weight,std=0.02)
        self.backbone=build_backbone(width,context)
        self.norm=nn.LayerNorm(width)
        self.out_bias=nn.Parameter(torch.zeros(vocab))
    def forward(self, tokens):
        h=self.norm(self.backbone(self.emb(tokens)))
        return F.linear(h,self.emb.weight,self.out_bias)  # tied LM output weights


# Attention uses native PyTorch SDPA, which dispatches supported CUDA kernels.
# No hand-written attention tensor BxHxTxT is constructed.
try:
    from torch.nn.attention import sdpa_kernel, SDPBackend
    HAVE_SDPA_CONTEXT=True
except ImportError:
    HAVE_SDPA_CONTEXT=False

SDPA_FUSED_AVAILABLE=False
SDPA_PROBE_NOTE='not tested'

def check_fused_sdpa(head_dim=16):
    """Confirm fused SDPA forward AND backward on the *actual* Kaggle GPU."""
    global SDPA_PROBE_NOTE
    if not HAVE_SDPA_CONTEXT:
        SDPA_PROBE_NOTE='sdpa_kernel context unavailable: automatic backend selection only'
        return False
    d=torch.float16 if USE_FP16_SDPA_ATTENTION else torch.float32
    q=torch.randn(2,4,128,head_dim,device=DEVICE,dtype=d,requires_grad=True)
    k=torch.randn_like(q,requires_grad=True);v=torch.randn_like(q,requires_grad=True)
    try:
        with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION,SDPBackend.EFFICIENT_ATTENTION]):
            y=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0)
            y.float().square().mean().backward()
        torch.cuda.synchronize()
        SDPA_PROBE_NOTE='fused SDPA forward+backward succeeded (exact Flash vs efficient choice is backend-selected)'
        return True
    except Exception as e:
        SDPA_PROBE_NOTE='fused SDPA not available for current input: '+str(e)[:250]
        return False

class FastCausalAttentionBlock(nn.Module):
    def __init__(self,width,num_heads,ffn_dim):
        super().__init__()
        assert width%num_heads==0
        self.nheads=num_heads;self.head_dim=width//num_heads
        self.ln1=nn.LayerNorm(width)
        self.qkv=nn.Linear(width,3*width)
        self.out=nn.Linear(width,width)
        self.ln2=nn.LayerNorm(width)
        self.ffn=nn.Sequential(nn.Linear(width,ffn_dim),nn.GELU(),nn.Linear(ffn_dim,width))
    def forward(self,x):
        b,t,c=x.shape
        q,k,v=self.qkv(self.ln1(x)).reshape(b,t,3,self.nheads,self.head_dim).unbind(2)
        q=q.transpose(1,2);k=k.transpose(1,2);v=v.transpose(1,2)
        if USE_FP16_SDPA_ATTENTION and x.device.type=='cuda':
            q,k,v=q.half(),k.half(),v.half()
        if SDPA_FUSED_AVAILABLE:
            with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION,SDPBackend.EFFICIENT_ATTENTION]):
                att=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=0.,is_causal=True)
        else:
            att=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=0.,is_causal=True)
        att=att.transpose(1,2).contiguous().view(b,t,c).to(x.dtype)
        x=x+self.out(att)
        return x+self.ffn(self.ln2(x))

class TransformerLM(nn.Module):
    def __init__(self,vocab,width,num_heads,ffn_dim,depth=DEPTH):
        super().__init__()
        self.vocab=vocab
        self.emb=nn.Embedding(vocab,width)
        nn.init.normal_(self.emb.weight,std=0.02)
        self.blocks=nn.ModuleList([FastCausalAttentionBlock(width,num_heads,ffn_dim) for _ in range(depth)])
        self.norm=nn.LayerNorm(width)
        self.out_bias=nn.Parameter(torch.zeros(vocab))
        # A position embedding is intentionally absent; use ALiBi/RoPE if desired in separate experiment.
        # Fixed/learned positional encodings are an important architectural confounder.
    def forward(self,tokens):
        x=self.emb(tokens)
        for block in self.blocks: x=block(x)
        x=self.norm(x)
        return F.linear(x,self.emb.weight,self.out_bias)

# IMPORTANT: For memory / order-sensitive language tasks, the transformer must have position information.
# We add causal ALiBi bias to SDPA as a deterministic, parameter-free relative-position scheme.
# The model source above will be adjusted before construction by applying the class below.
class FastCausalRotaryBlock(FastCausalAttentionBlock):
    def forward(self,x):
        b,t,c=x.shape
        q,k,v=self.qkv(self.ln1(x)).reshape(b,t,3,self.nheads,self.head_dim).unbind(2)
        q=q.transpose(1,2);k=k.transpose(1,2);v=v.transpose(1,2)
        if USE_FP16_SDPA_ATTENTION and x.device.type=='cuda':q,k,v=q.half(),k.half(),v.half()
        # Rotary embeddings provide relative position information and preserve fused causal SDPA support.
        q,k=apply_rotary(q,k)
        if SDPA_FUSED_AVAILABLE:
            with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION,SDPBackend.EFFICIENT_ATTENTION]):
                att=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=0.,is_causal=True)
        else:
            att=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=0.,is_causal=True)
        att=att.transpose(1,2).contiguous().view(b,t,c).to(x.dtype)
        x=x+self.out(att)
        return x+self.ffn(self.ln2(x))

def apply_rotary(q,k):
    # Standard pairwise rotary on head vectors, positions 0..T-1; no trainable parameters.
    d=q.shape[-1]
    if d%2:raise ValueError('Transformer head dim must be even for RoPE')
    pos=torch.arange(q.shape[-2],device=q.device,dtype=torch.float32)
    inv=1.0/(10000.0**(torch.arange(0,d,2,device=q.device,dtype=torch.float32)/d))
    ang=pos[:,None]*inv[None,:]
    cos=ang.cos().to(q.dtype)[None,None,:,:];sin=ang.sin().to(q.dtype)[None,None,:,:]
    def rot(x):
        even=x[...,0::2];odd=x[...,1::2]
        return torch.stack((even*cos-odd*sin,even*sin+odd*cos),dim=-1).flatten(-2)
    return rot(q),rot(k)

class RotaryTransformerLM(TransformerLM):
    def __init__(self,vocab,width,num_heads,ffn_dim,depth=DEPTH):
        super().__init__(vocab,width,num_heads,ffn_dim,depth)
        self.blocks=nn.ModuleList([FastCausalRotaryBlock(width,num_heads,ffn_dim) for _ in range(depth)])

def transformer_match(vocab,target,probe_classes=None):
    """Exactly count parameters; prefer head_dim=16 fused-friendly width when within 0.75%."""
    best=None
    for w in (64,80,72,96):
        for nhead in (4,):
            if w%nhead:continue
            for ffd in range(96,768,8):
                # Each residual block: two LN, QKV, output projection and 2-layer MLP.
                block=4*w*w+2*w*ffd+9*w+ffd
                head=(vocab if probe_classes is None else probe_classes*w+probe_classes)
                n=vocab*w+2*w+DEPTH*block+head
                gap=abs(n-target)
                # Matching to within 0.75% is sufficient; then favor width64/head_dim16.
                within=gap/target<=0.0075
                key=(0 if within and w==64 else 1 if within else 2,gap,abs(ffd-256))
                if best is None or key<best[0]:best=(key,w,nhead,ffd,n)
    _,w,nhead,ffd,n=best
    return {'width':w,'heads':nhead,'ffn':ffd,'params':n,'target':target,
            'param_difference_pct':100*(n-target)/target}


## 4. Data preparation: attached FineWeb-Edu, same split/tokenizer for both models

In [ ]:
"""Frozen FineWeb-Edu sample-10BT data pipeline.
Only the dataset/test *types* were taken from the provided example notebook.
No model code or experiment results from that notebook are imported.
"""
import glob,hashlib,json,os,time
from pathlib import Path
import numpy as np


def sha_text(x):
    return hashlib.sha256(x.encode('utf8','ignore')).hexdigest()


def split_doc(text):
    """Exact duplicates land in the same split; near-duplicates remain a caveat."""
    r=int(sha_text(text)[:12],16)%1000
    return 'valid' if r<10 else 'test' if r<20 else 'train'


def documents(paths):
    import pyarrow.parquet as pq
    for filename in paths:
        pqfile=pq.ParquetFile(str(filename))
        if 'text' not in pqfile.schema.names:
            raise ValueError(f'Not a FineWeb text Parquet shard: {filename}')
        for batch in pqfile.iter_batches(columns=['text'],batch_size=512):
            for raw in batch.column(0).to_pylist():
                if isinstance(raw,str) and len(raw.strip())>=20:
                    yield raw.replace('\x00',' ').strip()


def find_paths(pattern):
    found=[Path(p) for p in sorted(glob.glob(pattern,recursive=True)) if Path(p).is_file()]
    if not found:
        raise FileNotFoundError('FineWeb-Edu parquet not found. Attach sample/10BT/000_00000.parquet in Kaggle Inputs, then edit PARQUET_GLOB. Current: '+pattern)
    if len(found)>1:
        filtered=[p for p in found if 'fineweb' in str(p).lower()]
        if filtered:found=filtered
        else:raise RuntimeError('Multiple generic parquet shards matched. Narrow PARQUET_GLOB to FineWeb-Edu sample-10BT only.')
    return found


def manifest_for_input(paths):
    rows=[]
    for p in paths:
        with p.open('rb') as stream:head=stream.read(1024*1024)
        rows.append({'path':str(p),'file_size':p.stat().st_size,
             'first_megabyte_sha256':hashlib.sha256(head).hexdigest()})
    return rows


def prepare(root,glob_pattern,existing_data_root='',vocab=4096,max_train=30_000_000,
            valid_count=250_000,test_count=250_000,sample_chars=3_000_000,
            stop_epoch=None):
    import sentencepiece as spm
    out=Path(root)/'data';out.mkdir(parents=True,exist_ok=True)
    if existing_data_root:
        base=Path(existing_data_root)/'data'
        if not (base/'spm.model').exists() or not (base/'train.bin').exists():
            raise ValueError('EXISTING_DATA_ROOT must contain data/spm.model and train/valid/test.bin')
        manifest=json.loads((base/'manifest.json').read_text()) if (base/'manifest.json').exists() else {}
        return {'root':str(base),'tokenizer':str(base/'spm.model'),'sizes':{
            part:(base/f'{part}.bin').stat().st_size//2 for part in ('train','valid','test')},
            'reused':True,'original_manifest':manifest,'source':'FineWeb-Edu, externally prepared corpus (verify uploaded provenance)'}

    old=out/'manifest.json'
    if old.exists() and (out/'spm.model').exists():
        manifest=json.loads(old.read_text())
        if manifest.get('complete') and all((out/f'{k}.bin').exists() for k in ('train','valid','test')):
            return manifest
    paths=find_paths(glob_pattern)
    source_manifest=manifest_for_input(paths)
    sample=out/'spm_training_sample.txt'
    if not (out/'spm.model').exists():
        chars=0;count=0
        with sample.open('w',encoding='utf8') as f:
            for t in documents(paths):
                if split_doc(t)!='train':continue
                t=' '.join(t[:15000].split());f.write(t+'\n');chars+=len(t);count+=1
                if chars>=sample_chars:break
        if chars<20000:raise RuntimeError('Not enough training text to fit a tokenizer')
        spm.SentencePieceTrainer.Train(input=str(sample),model_prefix=str(out/'spm'),
            vocab_size=vocab,model_type='bpe',character_coverage=0.9995,
            unk_id=0,eos_id=1,bos_id=-1,pad_id=-1,
            max_sentence_length=16384,hard_vocab_limit=False, num_threads=4)
        sample.unlink(missing_ok=True)
    sp=spm.SentencePieceProcessor(model_file=str(out/'spm.model'))
    if sp.vocab_size()>65535:raise ValueError('uint16 token array requires vocab<=65535')
    sizes={'train':int(max_train),'valid':int(valid_count),'test':int(test_count)}
    mm={}
    for k,n in sizes.items():
        mm[k]=np.memmap(out/f'{k}.bin',dtype=np.uint16,mode='w+',shape=(n,))
    pos={k:0 for k in sizes}
    doc_count={k:0 for k in sizes}
    print('Tokenizing attached FineWeb-Edu. Caps:',sizes,flush=True)
    for j,t in enumerate(documents(paths)):
        if stop_epoch is not None and time.time()>stop_epoch:
            print('Corpus preparation budget reached; using filled prefixes, if sufficient.',flush=True)
            break
        split=split_doc(t)
        remain=sizes[split]-pos[split]
        if remain<=0:
            if all(pos[k]>=sizes[k] for k in sizes):break
            continue
        ids=sp.encode(t,out_type=int)
        if not ids:continue
        ids.append(sp.eos_id())
        take=min(len(ids),remain)
        mm[split][pos[split]:pos[split]+take]=np.asarray(ids[:take],dtype=np.uint16)
        pos[split]+=take;doc_count[split]+=1
        if j and j%5000==0:print('data',j,'documents',pos,flush=True)
    for k in sizes:
        mm[k].flush();del mm[k]
        with (out/f'{k}.bin').open('r+b') as f:f.truncate(pos[k]*2)
    if pos['train']<512*128 or min(pos['valid'],pos['test'])<4096:
        raise RuntimeError('Not enough authentic FineWeb-Edu text before preparation deadline: '+str(pos))
    manifest={'complete':True,'root':str(out),'tokenizer':str(out/'spm.model'),
        'dataset':'HuggingFaceFW/fineweb-edu sample-10BT','data_provenance':source_manifest,
        'document_hash_split':'1% validation, 1% test, 98% train; exact hash by full text',
        'document_counts':doc_count,'sizes':pos,'vocab_size':sp.vocab_size(),
        'reused':False,'tokens_are_bpe':True,
        'limits':sizes,'sha256_tokenizer':hashlib.sha256((out/'spm.model').read_bytes()).hexdigest()}
    old.write_text(json.dumps(manifest,indent=2))
    return manifest


def load_tokens(manifest,part):
    f=Path(manifest['root'])/f'{part}.bin'
    return np.memmap(f,mode='r',dtype=np.uint16)

In [ ]:
# Never fall back to Tiny Shakespeare silently.
def locate_prepared():
    bases=[]
    if PREPARED_DIR:bases.append(Path(PREPARED_DIR))
    for path in glob.glob('/kaggle/input/**/train.bin',recursive=True):bases.append(Path(path).parent)
    bases.append(OUT/'data')
    for folder in dict.fromkeys(bases):
        if all((folder/p).exists() for p in ('train.bin','valid.bin','test.bin','spm.model')):
            return folder
    return None

prepared=locate_prepared()
if prepared is not None:
    import sentencepiece as spm
    processor=spm.SentencePieceProcessor(model_file=str(prepared/'spm.model'))
    VOCAB=processor.vocab_size()
    TOKENS={p:np.memmap(prepared/f'{p}.bin',mode='r',dtype=np.uint16) for p in ('train','valid','test')}
    DATA_INFO={'mode':'reused_prepared','prepared':str(prepared),'sizes':{k:len(x) for k,x in TOKENS.items()},
               'tokenizer_sha256':hashlib.sha256((prepared/'spm.model').read_bytes()).hexdigest()}
else:
    parquet_paths=glob.glob(TRAIN_PARQUET_GLOB,recursive=True)
    if len(parquet_paths)!=1:
        raise FileNotFoundError(f'Attach Kaggle dataset cartman with one 000_00000.parquet; matched {len(parquet_paths)}: {parquet_paths[:6]}')
    manifest=prepare(str(OUT),TRAIN_PARQUET_GLOB,vocab=4096,max_train=MAX_PREP_TRAIN_TOKENS,
                     valid_count=250_000,test_count=250_000)
    TOKENS={p:load_tokens(manifest,p) for p in ('train','valid','test')}
    VOCAB=int(manifest['vocab_size'])
    DATA_INFO={'mode':'attached_fineweb_parquet','manifest':manifest,'sizes':{k:len(x) for k,x in TOKENS.items()}}
assert min(map(len,TOKENS.values()))>=CONTEXT+10000,'Not enough held-out tokens'
assert max(int(np.max(v)) for v in TOKENS.values()) < VOCAB,'Tokenizer/data mismatch'
DATA_INFO['fingerprint']=hashlib.sha256(
    np.asarray(TOKENS['train'][:524288],dtype=np.uint16).tobytes()
    +str(VOCAB).encode()).hexdigest()
write_json(OUT/'dataset_info.json',DATA_INFO)
print('FineWeb-Edu dataset:',DATA_INFO['mode'],'vocab',VOCAB,'tokens',DATA_INFO['sizes'])
print('200M tokens/model are token PRESENTATIONS; prepared distinct training tokens:',len(TOKENS['train']))

## 5. Model matching, numerical checks, CUDA Graphs and SDPA kernel check

In [ ]:
# Both models are FP32 except the Transformer's attention q/k/v when USE_FP16_SDPA_ATTENTION=True.
# No change to the original PPLSTM recurrence or initialization.
SDPA_FUSED_AVAILABLE=check_fused_sdpa(head_dim=16)
if FUSED_SDPA_ONLY and not SDPA_FUSED_AVAILABLE:raise RuntimeError('Requested fused attention only, but kernel is unavailable')
print('SDPA fused availability:',SDPA_FUSED_AVAILABLE,SDPA_PROBE_NOTE)
print('SDPA requested half attention:',USE_FP16_SDPA_ATTENTION)

# Important: original code's parameter count is measured at the actual Kaggle vocabulary.
HYBRID=PPLSTMLanguageModel(VOCAB).to(DEVICE)
MATCH=transformer_match(VOCAB,nparams(HYBRID))
TRANSFORMER=RotaryTransformerLM(VOCAB,MATCH['width'],MATCH['heads'],MATCH['ffn']).to(DEVICE)
print('Parameter match:',{'hybrid':nparams(HYBRID),'transformer':nparams(TRANSFORMER),**MATCH})
assert abs(MATCH['param_difference_pct'])<1.5,'Could not match parameter counts within 1.5%'

TESTS=[]
def test_row(name,ok,**kwargs):
    obj={'test':name,'passed':bool(ok),**kwargs};TESTS.append(obj)
    save_table('tests.csv',TESTS);print('PASS' if ok else 'FAIL',name,kwargs)
    if not ok:raise RuntimeError('Required correctness check failed: '+name)

# Exact recurrence: parallel vs serial and gradients. This uses original PPLSTM implementation.
cell=PPLSTM(12).to(DEVICE,dtype=torch.float64)
x=torch.randn(2,17,12,device=DEVICE,dtype=torch.float64,requires_grad=True)
y1=cell(x);y2=cell.serial_reference(x)
err=(y1-y2).abs().max().item()
g1=torch.autograd.grad(y1.square().mean(),(x,*cell.parameters()),retain_graph=True)
g2=torch.autograd.grad(y2.square().mean(),(x,*cell.parameters()))
ge=max((a-b).abs().max().item() for a,b in zip(g1,g2))
test_row('PPLSTM_serial_parallel_float64',err<1e-9 and ge<1e-9,output_max_diff=err,gradient_max_diff=ge)
del cell,x,y1,y2,g1,g2

# Ensure both architectures are causal at the token-logit level.
for name,m in [('hybrid',HYBRID),('transformer',TRANSFORMER)]:
    m.eval()
    with torch.no_grad():
        a=torch.randint(0,VOCAB,(1,48),device=DEVICE);b=a.clone()
        b[:,24:]=torch.randint(0,VOCAB,(1,24),device=DEVICE)
        ea=(m(a)[:,:24]-m(b)[:,:24]).abs().max().item()
    test_row(name+'_causality',ea<1e-4,max_error=ea)
    m.train()

# CUDA Graph captures fixed-shape forward+backward; AdamW and gradient clipping are outside.
# Training & evaluation execute all model parameters in FP32, apart from optional attention QKV tensors.
class ForwardBackwardGraph:
    def __init__(self,model,example_x,example_y):
        self.model=model
        self.static_x=torch.zeros_like(example_x)
        self.static_y=torch.zeros_like(example_y)
        self.graph=torch.cuda.CUDAGraph()
        stream=torch.cuda.Stream(device=DEVICE)
        stream.wait_stream(torch.cuda.current_stream())
        model.train()
        with torch.cuda.stream(stream):
            for _ in range(3):
                model.zero_grad(set_to_none=True)
                self.compute_loss().backward()
        torch.cuda.current_stream().wait_stream(stream)
        model.zero_grad(set_to_none=True)
        with torch.cuda.graph(self.graph,stream=stream):
            self.loss=self.compute_loss()
            self.loss.backward()
    def compute_loss(self):
        logits=self.model(self.static_x)
        return F.cross_entropy(logits.reshape(-1,logits.shape[-1]),self.static_y.reshape(-1),ignore_index=-100)
    def replay(self,x,y):
        assert x.shape==self.static_x.shape and y.shape==self.static_y.shape
        self.static_x.copy_(x);self.static_y.copy_(y)
        self.graph.replay()
        return self.loss

def step_update(runner,x,y):
    model=runner['model'];optimizer=runner['opt']
    if runner['graph'] is not None:
        loss=runner['graph'].replay(x,y)
    else:
        optimizer.zero_grad(set_to_none=True)
        logits=model(x)
        loss=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),y.reshape(-1),ignore_index=-100)
        loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    optimizer.step()
    return loss


def lm_batch(step,count=BATCH*CONTEXT):
    # Both runners call this using the same model-specific step index.
    rng=np.random.default_rng(SEED+98765431*int(step))
    arr=TOKENS['train'];max_start=(len(arr)-CONTEXT-1)//CONTEXT
    ids=rng.integers(0,max_start,size=BATCH)
    w=np.stack([np.asarray(arr[int(j)*CONTEXT:int(j)*CONTEXT+CONTEXT+1],dtype=np.int64) for j in ids])
    x=torch.from_numpy(w[:,:-1].copy()).to(DEVICE)
    y=torch.from_numpy(w[:,1:].copy()).to(DEVICE)
    if count<BATCH*CONTEXT:y.reshape(-1)[count:]=-100
    return x,y

def make_optimizer(m):
    # PyTorch fused AdamW is optional; it is not xLSTM's optimized CUDA kernel.
    try:return torch.optim.AdamW(m.parameters(),lr=8e-4,betas=(0.9,0.95),weight_decay=.01,fused=True)
    except (RuntimeError,TypeError):return torch.optim.AdamW(m.parameters(),lr=8e-4,betas=(0.9,0.95),weight_decay=.01)

RUNNERS={}
for name,m in [('hybrid_pplstm',HYBRID),('transformer_sdpa',TRANSFORMER)]:
    opt=make_optimizer(m)
    ex_x,ex_y=lm_batch(0)
    try:
        graph=ForwardBackwardGraph(m,ex_x,ex_y)
        engine='cuda_graph_forward_backward'
    except Exception as exc:
        if name=='hybrid_pplstm' and PPLSTM_CUDA_GRAPH_REQUIRED:raise RuntimeError('Required hybrid CUDA Graph capture failed') from exc
        if name=='transformer_sdpa' and TRANSFORMER_CUDA_GRAPH_REQUIRED:raise RuntimeError('Required Transformer graph capture failed') from exc
        print('CUDA Graph unavailable',name,repr(exc),'running eager, explicitly labelled')
        graph=None;engine='eager'
    RUNNERS[name]={'model':m,'opt':opt,'graph':graph,'engine':engine,'seen':0,'steps':0,'train_seconds':0.0,
                   'curve':[]}
    test_row(name+'_graph_capture_or_eager_fallback',True,engine=engine)

# Forward/backward and optimizer smoke steps, with checks for finite loss AND gradients.
for name,r in RUNNERS.items():
    xx,yy=lm_batch(0)
    loss=step_update(r,xx,yy)
    torch.cuda.synchronize()
    allgood=math.isfinite(float(loss.item())) and all(p.grad is None or bool(torch.isfinite(p.grad).all().item()) for p in r['model'].parameters())
    test_row(name+'_training_step_finite',allgood,loss=float(loss.item()))
    # We count this smoke update as the first actual training step.
    r['steps']=1;r['seen']=BATCH*CONTEXT

write_json(OUT/'config.json',{'target_tokens_per_model':TARGET_TOKENS_PER_MODEL,'width_hybrid':WIDTH,'depth':DEPTH,
 'context':CONTEXT,'batch':BATCH,'transformer_match':MATCH,'seed':SEED,
 'model_precision':'float32 parameters and layers',
 'sdpa_attention_dtype':'fp16 qkv within SDPA only' if USE_FP16_SDPA_ATTENTION else 'fp32',
 'sdpa_fused_available':SDPA_FUSED_AVAILABLE,'sdpa_probe_note':SDPA_PROBE_NOTE,
 'model_engines':{name:r['engine'] for name,r in RUNNERS.items()},
 'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,
 'dataset_fingerprint':DATA_INFO['fingerprint'],
 'limitations':['single seed','different architectures, matched parameter count only',
 'CUDA Graph captures forward+backward; optimizer outside','attention FP16 if enabled',
 'PPLSTM tested against Transformer, NOT optimized sLSTM',
 'synthetic task probes use independently trained models']})

## 6. Same-data accuracy, fair alternating training, speed pilot and resumable checkpoints

In [ ]:
# Held-out validation is identical for both models; tied embeddings, same input sampler and schedule.
@torch.no_grad()
def evaluate_language(r,split='valid',limit=SMALL_VALIDATION_TOKENS):
    m=r['model'];m.eval();arr=TOKENS[split]
    windows=min(limit//CONTEXT,(len(arr)-1)//CONTEXT)
    assert windows>0
    nll=0.0;correct=0;seen=0
    for s in range(0,windows,BATCH):
        nb=min(BATCH,windows-s)
        w=np.stack([np.asarray(arr[(s+j)*CONTEXT:(s+j)*CONTEXT+CONTEXT+1],dtype=np.int64) for j in range(nb)])
        x=torch.from_numpy(w[:,:-1].copy()).to(DEVICE)
        y=torch.from_numpy(w[:,1:].copy()).to(DEVICE)
        scores=m(x)
        nll+=float(F.cross_entropy(scores.reshape(-1,VOCAB),y.reshape(-1),reduction='sum').item())
        correct+=int((scores.argmax(-1)==y).sum().item());seen+=y.numel()
    m.train()
    return {'n_tokens':seen,'val_nll':nll/seen,'val_ppl':math.exp(nll/seen),'top1_pct':100*correct/seen}

def learning_rate(seen):
    warmup=1_000_000;peak=8e-4;minimum=8e-5
    if seen<warmup:return peak*max(1,seen+CONTEXT)/warmup
    pct=min(1.,max(0.,(seen-warmup)/max(1,TARGET_TOKENS_PER_MODEL-warmup)))
    return minimum+(peak-minimum)*.5*(1+math.cos(math.pi*pct))

ALL_CURVE=[]
RUNTIME_META=[]

def read_resume(r,name):
    candidates=[]
    if RESUME_INPUT_DIR:
        candidates.extend(glob.glob(str(Path(RESUME_INPUT_DIR)/f'{name}_checkpoint.pt')))
        candidates.extend(glob.glob(str(Path(RESUME_INPUT_DIR)/'**'/f'{name}_checkpoint.pt'),recursive=True))
    local=OUT/f'{name}_checkpoint.pt'
    if local.exists():candidates.append(str(local))
    if not candidates:return
    checkpoint=torch.load(candidates[0],map_location=DEVICE,weights_only=False)
    for key,expected in [('fingerprint',DATA_INFO['fingerprint']),('vocab',VOCAB),('context',CONTEXT),('target',TARGET_TOKENS_PER_MODEL)]:
        assert checkpoint.get(key)==expected,f'Cannot resume {name}: {key} mismatch'
    # If a new run resumes, the uncounted smoke step is overwritten by saved parameters.
    r['model'].load_state_dict(checkpoint['model']);r['opt'].load_state_dict(checkpoint['optimizer'])
    r['seen']=int(checkpoint['seen']);r['steps']=int(checkpoint['steps']);r['train_seconds']=float(checkpoint['train_seconds'])
    r['curve']=list(checkpoint.get('curve',[]))
    print('RESUMED',name,'from',candidates[0],'tokens',r['seen'])
    # Rebuild captured graph after loading model/optimizer states (capture references model tensors).
    del r['graph'];r['graph']=None
    x,y=lm_batch(r['steps'])
    try:
        r['graph']=ForwardBackwardGraph(r['model'],x,y);r['engine']='cuda_graph_forward_backward'
    except Exception as ex:
        if name=='hybrid_pplstm' and PPLSTM_CUDA_GRAPH_REQUIRED:raise RuntimeError('Resume graph recapture failed') from ex
        r['engine']='eager'
    return True

for name,r in RUNNERS.items():read_resume(r,name)

def checkpoint(name,r):
    atomic_save({'model':r['model'].state_dict(),'optimizer':r['opt'].state_dict(),
       'seen':r['seen'],'steps':r['steps'],'train_seconds':r['train_seconds'],'curve':r['curve'],
       'fingerprint':DATA_INFO['fingerprint'],'vocab':VOCAB,'context':CONTEXT,
       'target':TARGET_TOKENS_PER_MODEL,'engine':r['engine']},OUT/f'{name}_checkpoint.pt')

def refresh_curve(name,r,force=False):
    if r['seen']==0:return
    if not force and r['seen']<VALIDATE_EVERY: return
    metrics=evaluate_language(r,limit=SMALL_VALIDATION_TOKENS)
    row={'model':name,'tokens_presented':r['seen'],'steps':r['steps'],
         'train_seconds_excluding_validation':r['train_seconds'],
         'tokens_per_second':r['seen']/max(r['train_seconds'],1e-7),
         'engine':r['engine'],**metrics}
    r['curve'].append(row)
    all_rows=[x for each in RUNNERS.values() for x in each['curve']]
    save_table('language_learning_curve.csv',all_rows)
    print(name,'TOKENS',format(r['seen'],','),'PPL',round(metrics['val_ppl'],2),
          'ACC',round(metrics['top1_pct'],2),'tok/s',round(row['tokens_per_second']),flush=True)
    bundle()

# Short pilot is INCLUDED in token budgets, not extra optimizer updates.
PILOT_STEPS=50
for name,r in RUNNERS.items():
    t0=time.perf_counter()
    for _ in range(PILOT_STEPS):
        if r['seen']>=TARGET_TOKENS_PER_MODEL:break
        cnt=min(CONTEXT*BATCH,TARGET_TOKENS_PER_MODEL-r['seen'])
        x,y=lm_batch(r['steps'],cnt)
        for pg in r['opt'].param_groups:pg['lr']=learning_rate(r['seen'])
        loss=step_update(r,x,y)
        r['steps']+=1;r['seen']+=cnt
    torch.cuda.synchronize()
    dt=time.perf_counter()-t0
    r['train_seconds']+=dt
    rate=(PILOT_STEPS*BATCH*CONTEXT)/max(dt,1e-9)
    RUNTIME_META.append({'model':name,'pilot_steps':PILOT_STEPS,'pilot_tokens_per_second':rate,
                         'full_target_hours_estimate':TARGET_TOKENS_PER_MODEL/max(rate,1e-9)/3600,
                         'engine':r['engine'],'pilot_loss':float(loss.item())})
    print('PILOT',name,'tok/s',round(rate),'estimated hours for 200M',round(TARGET_TOKENS_PER_MODEL/rate/3600,2))
save_table('speed_pilot.csv',RUNTIME_META)
print('ESTIMATED COMBINED TRAIN HOURS',round(sum(row['full_target_hours_estimate'] for row in RUNTIME_META),2),
      'hours; time guard:',MAX_RUN_HOURS,'hours incl dataset setup/probes.')
bundle()

# Each model sees exactly the same batch at the same step index.
# Alternating training stops one model from consuming the entire quota first.
if not STARTUP_ONLY:
    last_checkpoint={k:r['seen']//SAVE_EVERY for k,r in RUNNERS.items()}
    # A chunk for each model, lower-token model first.
    while min(r['seen'] for r in RUNNERS.values())<TARGET_TOKENS_PER_MODEL:
        # Budget guard checks wall-clock and finishes at a chunk boundary.
        if time.monotonic() > DEADLINE-900:
            print('STOP: 15-minute buffer before wall-time budget. Checkpoints/export retained.')
            break
        name=min(RUNNERS, key=lambda k:RUNNERS[k]['seen'])
        r=RUNNERS[name]
        if r['seen']>=TARGET_TOKENS_PER_MODEL:break
        chunk_end=min(TARGET_TOKENS_PER_MODEL,r['seen']+CHUNK_TOKENS)
        next_report=((r['seen']//VALIDATE_EVERY)+1)*VALIDATE_EVERY
        chunk_end=min(chunk_end,next_report)
        t0=time.perf_counter()
        while r['seen']<chunk_end:
            # Check every 100 steps to preserve quota, then export partial results.
            if (r['steps']%100==0) and time.monotonic()>DEADLINE-900:
                print('Time guard hit during chunk',name);break
            cnt=min(BATCH*CONTEXT,chunk_end-r['seen'])
            x,y=lm_batch(r['steps'],cnt)
            lr=learning_rate(r['seen'])
            for pg in r['opt'].param_groups:pg['lr']=lr
            loss=step_update(r,x,y)
            r['steps']+=1;r['seen']+=cnt
            if (r['steps']%1000)==0 and not math.isfinite(float(loss.item())):
                raise FloatingPointError(f'Nonfinite loss {name} step {r["steps"]}')
        torch.cuda.synchronize()
        r['train_seconds']+=time.perf_counter()-t0
        if r['seen']>=next_report or r['seen']==TARGET_TOKENS_PER_MODEL:
            refresh_curve(name,r,force=True)
        if r['seen']//SAVE_EVERY>last_checkpoint[name] or r['seen']==TARGET_TOKENS_PER_MODEL or time.monotonic()>DEADLINE-900:
            checkpoint(name,r)
            last_checkpoint[name]=r['seen']//SAVE_EVERY
            bundle()
            print('SAVED',name,'at',format(r['seen'],','),'tokens')
        if time.monotonic()>DEADLINE-900:break

# Final checkpoint and evaluation, even if run is partial. Both models are preserved.
FINALS=[]
for name,r in RUNNERS.items():
    checkpoint(name,r)
    valid=evaluate_language(r,'valid',FINAL_EVAL_TOKENS)
    test=evaluate_language(r,'test',FINAL_EVAL_TOKENS)
    FINALS.append({'model':name,'status':'complete' if r['seen']>=TARGET_TOKENS_PER_MODEL else 'partial',
          'tokens_presented':r['seen'],'steps':r['steps'],'parameters':nparams(r['model']),
          'train_seconds':r['train_seconds'],'train_tok_per_s':r['seen']/max(r['train_seconds'],1e-8),
          'engine':r['engine'],
          **{'final_valid_'+k:v for k,v in valid.items()},
          **{'final_test_'+k:v for k,v in test.items()}})
    print('FINAL',FINALS[-1])
save_table('language_final.csv',FINALS)
write_json(OUT/'language_summary.json',FINALS)
print('MODEL TRAINING ENDED. Results:',bundle())

## 7. Optional synthetic probes: train both architectures on fresh labelled tasks

In [ ]:
# These independently trained classifiers do NOT evaluate the language model zero-shot.
# Fresh examples are generated on every optimizer step; held-out evaluation uses separate seeds.
# All tasks share identical token generator, train length, batch sizes, steps, and label spaces.

TASK_SPECS={
 'parity':2,'state_mod4':4,'delayed_recall':6,
 'key_value_recall':8,'key_value_update':8,'two_hop':8,'three_hop':8,
}
TASK_VOCAB=96

def make_probe_sequences(task,seq_len,batch,seed):
    rng=np.random.default_rng(seed)
    x=np.zeros((batch,seq_len),dtype=np.int64)
    y=np.zeros(batch,dtype=np.int64)
    for i in range(batch):
        if task=='parity':
            bits=rng.integers(0,2,size=seq_len-1)
            x[i,:-1]=bits+1;x[i,-1]=3
            y[i]=int(bits.sum()%2)
        elif task=='state_mod4':
            events=rng.integers(0,4,size=seq_len-1)
            state=0
            for ev in events:
                if ev==0:state=(state+1)%4
                elif ev==1:state=(state-1)%4
                elif ev==2:state=0
            x[i,:-1]=events+4;x[i,-1]=8;y[i]=state
        elif task=='delayed_recall':
            val=int(rng.integers(0,6));x[i,0]=20+val
            x[i,1:-1]=rng.integers(26,32,size=seq_len-2);x[i,-1]=32;y[i]=val
        elif task in ('key_value_recall','key_value_update'):
            # Eight keys, random values per example; no static mapping to memorize.
            keys=rng.permutation(8);values=rng.integers(0,8,size=8)
            records=[int(z) for pair in zip(40+keys,56+values) for z in pair]
            table={int(k):int(v) for k,v in zip(keys,values)}
            if task=='key_value_update':
                for _ in range(8):
                    k=int(rng.integers(0,8));v=int(rng.integers(0,8))
                    records += [40+k,56+v];table[k]=v
            q=int(rng.integers(0,8));y[i]=table[q]
            assert len(records)+2<=seq_len
            x[i,:len(records)]=records
            if seq_len-len(records)-2>0:x[i,len(records):-2]=rng.integers(72,80,size=seq_len-len(records)-2)
            x[i,-2]=40+q;x[i,-1]=80
        elif task in ('two_hop','three_hop'):
            hops=2 if task=='two_hop' else 3
            mapping=rng.permutation(8);start=int(rng.integers(0,8));v=start
            for _ in range(hops):v=int(mapping[v])
            y[i]=v
            records=[]
            for src in rng.permutation(8):records.extend([64+int(src),48+int(mapping[src])])
            assert len(records)+3<=seq_len
            x[i,:len(records)]=records
            x[i,len(records):-3]=rng.integers(72,80,size=seq_len-len(records)-3)
            x[i,-3:]=[64+start,81,hops+82]
        else:raise ValueError(task)
    return (torch.from_numpy(x).to(DEVICE),torch.from_numpy(y).to(DEVICE))

class HybridProbe(nn.Module):
    def __init__(self,classes):
        super().__init__();self.emb=nn.Embedding(TASK_VOCAB,WIDTH)
        self.backbone=build_backbone(WIDTH,max(PROBE_EVAL_CONTEXTS))
        self.norm=nn.LayerNorm(WIDTH);self.head=nn.Linear(WIDTH,classes)
    def forward(self,x):return self.head(self.norm(self.backbone(self.emb(x))[:,-1]))

class RotaryTransformerProbe(nn.Module):
    def __init__(self,vocab,classes,width,heads,ffn):
        super().__init__();self.emb=nn.Embedding(vocab,width)
        self.blocks=nn.ModuleList([FastCausalRotaryBlock(width,heads,ffn) for _ in range(DEPTH)])
        self.norm=nn.LayerNorm(width);self.head=nn.Linear(width,classes)
    def forward(self,x):
        h=self.emb(x)
        for block in self.blocks:h=block(h)
        return self.head(self.norm(h[:,-1]))

PROBE_ROWS=[];PROBE_CURVES=[]
@torch.no_grad()
def eval_probe(model,task,length,n_examples=PROBE_EVAL_EXAMPLES):
    model.eval();right=0;total=0;loss_total=0.
    for idx in range(0,n_examples,PROBE_BATCH):
        count=min(PROBE_BATCH,n_examples-idx)
        xx,yy=make_probe_sequences(task,length,count,seed=4321987+100_007*length+idx)
        scores=model(xx)
        right+=int((scores.argmax(1)==yy).sum().item())
        loss_total+=float(F.cross_entropy(scores,yy,reduction='sum').item());total+=count
    model.train();return {'accuracy_pct':100*right/total,'validation_loss':loss_total/total,'examples':total}

for task,classes in TASK_SPECS.items():
    if not RUN_PROBES or STARTUP_ONLY:break
    for arch in ('hybrid_pplstm','transformer_sdpa'):
        if time.monotonic()>DEADLINE-900:
            print('STOP probes: time budget. Export partial probe results.')
            break
        # Use deterministic seeds; fresh model per task.
        torch.manual_seed(SEED+list(TASK_SPECS).index(task)*101)
        if arch=='hybrid_pplstm':pm=HybridProbe(classes).to(DEVICE);p_match=None
        else:
            torch.manual_seed(SEED+list(TASK_SPECS).index(task)*101)
            ref=HybridProbe(classes)
            p_match=transformer_match(TASK_VOCAB,nparams(ref),probe_classes=classes)
            del ref
            pm=RotaryTransformerProbe(TASK_VOCAB,classes,p_match['width'],p_match['heads'],p_match['ffn']).to(DEVICE)
        op=make_optimizer(pm)
        # Same training setup; use eager for probes to avoid making custom graph fail a scientific accuracy result.
        # Both official hybrid and SDPA attention may run faster with graph in main experiment.
        t0=time.perf_counter();steps_done=0
        for i in range(PROBE_STEPS):
            if i%50==0 and time.monotonic()>DEADLINE-900:break
            xb,yb=make_probe_sequences(task,PROBE_TRAIN_CONTEXT,PROBE_BATCH,seed=SEED+17171*i)
            op.zero_grad(set_to_none=True)
            loss=F.cross_entropy(pm(xb),yb)
            loss.backward();torch.nn.utils.clip_grad_norm_(pm.parameters(),1.0);op.step()
            steps_done=i+1
            if steps_done%PROBE_REPORT_EVERY==0 or steps_done==PROBE_STEPS:
                ev=eval_probe(pm,task,PROBE_TRAIN_CONTEXT)
                PROBE_CURVES.append({'task':task,'model':arch,'step':steps_done,'context':PROBE_TRAIN_CONTEXT,
                                    'accuracy_pct':ev['accuracy_pct'],'validation_loss':ev['validation_loss'],
                                    'train_loss':float(loss.item())})
                save_table('probe_learning_curves.csv',PROBE_CURVES)
                print('PROBE',task,arch,steps_done,'train-context val%',round(ev['accuracy_pct'],2),flush=True)
        torch.cuda.synchronize();secs=time.perf_counter()-t0
        for length in PROBE_EVAL_CONTEXTS:
            ev=eval_probe(pm,task,length)
            PROBE_ROWS.append({'task':task,'model':arch,'context':length,'trained_at_context':PROBE_TRAIN_CONTEXT,
                'steps':steps_done,'params':nparams(pm),'matched_params_delta_pct':None if p_match is None else p_match['param_difference_pct'],
                'accuracy_pct':ev['accuracy_pct'],'validation_loss':ev['validation_loss'],'eval_examples':ev['examples'],
                'chance_pct':100/classes,'probe_train_seconds_excluding_final_eval':secs,
                'status':'completed' if steps_done==PROBE_STEPS else 'partial'})
        save_table('probe_results.csv',PROBE_ROWS)
        bundle()
        del pm,op;torch.cuda.empty_cache()
    if time.monotonic()>DEADLINE-900:break
print('PROBE FINISHED',len(PROBE_ROWS),'rows')

## 8. Export and download

In [ ]:
finish={'results':FINALS,'parameter_match':MATCH,'sdpa':{'requested_fp16_attention':USE_FP16_SDPA_ATTENTION,
        'fused_available':SDPA_FUSED_AVAILABLE,'probe_message':SDPA_PROBE_NOTE},
        'time_budget_hours':MAX_RUN_HOURS,'actual_notebook_wall_hours':(time.monotonic()-RUN_START)/3600,
        'probe_tasks':list(TASK_SPECS),'probe_evaluations':len(PROBE_ROWS),
        'notes':['200M is tokens presented PER MODEL, not unique text',
            'Main language experiment has two independent models, the same data sampler/optimizer',
            'Kaggle attached corpus and run details in dataset_info.json/config.json',
            'The Transformer uses PyTorch scaled_dot_product_attention with RoPE positions',
            'Fused attention may be selected automatically; check sdpa_probe_note',
            'The recurrent model uses CUDA Graph forward+backward; optimizer is outside',
            'Synthetic probes are separately trained, not zero-shot language model evaluations',
            'The run may be partial if time-budget guard fired']}
write_json(OUT/'summary.json',finish)
print('FINAL SCORES:')
display(pd.DataFrame(FINALS))
if PROBE_ROWS:
    display(pd.DataFrame(PROBE_ROWS)[['task','model','context','steps','accuracy_pct','chance_pct','params']])
archive=bundle()
print('\nDOWNLOAD: ',archive)
print('SAVED MODEL/OPTIMIZER CHECKPOINTS (NOT IN ZIP):')
for name in RUNNERS:print(OUT/f'{name}_checkpoint.pt')
print('\nNote: Kaggle can save outputs with Save Version → Save & Run All. Attach prior outputs and set RESUME_INPUT_DIR to resume.')

## 9. Reading results, warnings and data interpretation

Download `PPLSTM_vs_Transformer_FineWeb_200M_Results.zip` from the notebook Output folder. It includes the CSV learning curves, speed trial, diagnostic checks, final comparisons, synthetic tasks and dataset hash. To **resume**, save both `*_checkpoint.pt` files separately (the ZIP excludes weights), attach the earlier Kaggle notebook output as an Input and set `RESUME_INPUT_DIR`.

**Not an optimized xLSTM sLSTM comparison.** This experiment deliberately excludes vanilla/optimized sLSTM to conserve GPU time. It measures your hybrid against a comparable-size Transformer. Unlike FlashAttention's public API, `torch.nn.functional.scaled_dot_product_attention` lets PyTorch choose a supported fast kernel. The notebook tests if fused forward and backward actually work; Fused SDPA is enforced when the kernel probe succeeds; `FUSED_SDPA_ONLY=True` prevents a math fallback when it fails.

**Fairness and limits:** Approximately equal trainable parameter counts and identical data/optimizer steps, but different architectures, 72-wide recurrent representation versus auto-sized Transformer, and optionally FP16 *inside Transformer attention* while other operations remain FP32. This precision difference is logged. One seed and limited model scale mean no strong superiority claim. FineWeb-Edu token **presentations** may repeat prepared text. Factual/multi-hop synthetic probes have explicit labels, not a direct evaluation of factual knowledge learned from FineWeb-Edu.

**Time:** Whether both models can finish 200M each in the remaining Kaggle allowance depends on real pilot throughput, dataset preparation and GPU availability. The runtime guard intentionally saves partial results rather than promising completion.